In [40]:
import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.datasets import fetch_openml
from sklearn.metrics import mean_squared_error
%matplotlib inline

In [36]:
#Gather data
load_boston = fetch_openml(data_id=531)
boston_dataset = load_boston
data = pd.DataFrame(data=boston_dataset.data, columns=boston_dataset.feature_names)
data['CHAS'] = data['CHAS'].astype(float)
data['RAD'] = data['RAD'].astype(float)
features = data.drop(['INDUS','AGE'], axis=1)
log_prices = np.log(boston_dataset.target)
target = log_prices.to_frame('PRICE')
features.head()

C:\Users\ryan6\anaconda3\Lib\site-packages\sklearn\datasets\_openml.py:1002: FutureWarning: The default value of `parser` will change from `'liac-arff'` to `'auto'` in 1.4. You can set `parser='auto'` to silence this warning. Therefore, an `ImportError` will be raised from 1.4 if the dataset is dense and pandas is not installed. Note that the pandas parser may return different data types. See the Notes Section in fetch_openml's API doc for details.
  warn(


,CRIM,ZN,CHAS,NOX,RM,DIS,RAD,TAX,PTRATIO,B,LSTAT
0,0.00632,18.0,0.0,0.538,6.575,4.0900,1.0,296.0,15.3,396.90,4.98
1,0.02731,0.0,0.0,0.469,6.421,4.9671,2.0,242.0,17.8,396.90,9.14
2,0.02729,0.0,0.0,0.469,7.185,4.9671,2.0,242.0,17.8,392.83,4.03
3,0.03237,0.0,0.0,0.458,6.998,6.0622,3.0,222.0,18.7,394.63,2.94
4,0.06905,0.0,0.0,0.458,7.147,6.0622,3.0,222.0,18.7,396.90,5.33


In [37]:
CRIME_IDX = 0
ZN_IDX = 1
CHAS_IDX = 2
RM_IDX = 4
PTRATIO_IDX = 8


# property_stats = np.ndarray(shape=(1,11))
# property_stats[0][CRIME_IDX] = features['CRIM'].mean()
# property_stats[0][ZN_IDX] = features['ZN'].mean()
# property_stats[0][CHAS_IDX] = features['CHAS'].mean()
property_stats = features.mean().values.reshape(1,11)

In [16]:
type(features.mean())

pandas.core.series.Series

In [17]:
type(features.mean().values)

numpy.ndarray

In [18]:
features.mean().values.reshape(1,11)

array([[3.61352356e+00, 1.13636364e+01, 6.91699605e-02, 5.54695059e-01,
        6.28463439e+00, 3.79504269e+00, 9.54940711e+00, 4.08237154e+02,
        1.84555336e+01, 3.56674032e+02, 1.26530632e+01]])

In [43]:
regr = LinearRegression().fit(features, target)
fitted_vals = regr.predict(features)
MSE = mean_squared_error(target, fitted_vals)
RMSE = np.sqrt(MSE)
print(f'MSE: {round(MSE,3)} & RMSE: {round(RMSE,3)}')

MSE: 0.035 & RMSE: 0.188


In [59]:
def get_log_estimate(nr_rooms, students_per_classroom, next_to_river = 0.0, high_confidence = True):
    #Configure Property:
    property_stats[0][RM_IDX] = nr_rooms
    property_stats[0][CHAS_IDX] = next_to_river
    property_stats[0][PTRATIO_IDX] = students_per_classroom
    
    #Make Prdiction:
    log_estimate = regr.predict(property_stats)[0][0]
    
    #Calculate Range:
    if high_confidence:
        upper_bound = log_estimate + 2*RMSE
        lower_bound = log_estimate - 2*RMSE
        interval = 95
    else:
        upper_bound = log_estimate + RMSE
        lower_bound = log_estimate - RMSE
        interval = 68
    
    return log_estimate, upper_bound, lower_bound, interval

In [62]:
get_log_estimate(3,20,1.0, high_confidence=True)

C:\Users\ryan6\anaconda3\Lib\site-packages\sklearn\base.py:464: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


(2.776758191480404, 3.1517824618746646, 2.4017339210861435, 95)

In [68]:
Median_Price = np.median(boston_dataset.target)

In [66]:
def get_price_estimate(nr_rooms, students_per_classroom, next_to_river = 0.0, high_confidence = True):
    #Configure Property:
    property_stats[0][RM_IDX] = nr_rooms
    property_stats[0][CHAS_IDX] = next_to_river
    property_stats[0][PTRATIO_IDX] = students_per_classroom
    
    #Make Prdiction:
    log_estimate = regr.predict(property_stats)[0][0]
    
    #Calculate Range:
    if high_confidence:
        upper_bound = log_estimate + 2*RMSE
        lower_bound = log_estimate - 2*RMSE
        interval = 95
    else:
        upper_bound = log_estimate + RMSE
        lower_bound = log_estimate - RMSE
        interval = 68
    
    return np.e**log_estimate*1000, np.e**upper_bound*1000, np.e**lower_bound*1000, interval

In [67]:
get_price_estimate(3,20,1.0, high_confidence=True)

C:\Users\ryan6\anaconda3\Lib\site-packages\sklearn\base.py:464: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


(16066.850777132342, 23377.697319724903, 11042.306278678256, 95)

In [69]:
ZILLOW_PRICE_MEDIAN = 583.3
SCALE_FACTOR = ZILLOW_PRICE_MEDIAN / Median_Price

In [73]:
def get_current_price_estimate(nr_rooms, students_per_classroom, next_to_river = 0.0, high_confidence = True):
    #Configure Property:
    property_stats[0][RM_IDX] = nr_rooms
    property_stats[0][CHAS_IDX] = next_to_river
    property_stats[0][PTRATIO_IDX] = students_per_classroom
    
    #Make Prdiction:
    log_estimate = regr.predict(property_stats)[0][0]
    
    #Calculate Range:
    if high_confidence:
        upper_bound = log_estimate + 2*RMSE
        lower_bound = log_estimate - 2*RMSE
        interval = 95
    else:
        upper_bound = log_estimate + RMSE
        lower_bound = log_estimate - RMSE
        interval = 68
    
    return np.e**log_estimate*SCALE_FACTOR*1000, np.e**upper_bound*SCALE_FACTOR*1000, np.e**lower_bound*SCALE_FACTOR*1000, interval

In [97]:
def get_dollar_estimate(rm, ptratio,chas=0.0,large_range=True):
    """Estimate the price of a property in Boston
    Keyword arguments:
    rm - number of rooms in the property
    ptratio - Student to teacher ratio
    chas - 1.0 if property next to Chas river
    large_range - True by default, giving 95% confidence interval, else gives 68% interval
    
    """
    if rm < 1 or ptratio < 1:
        print('That is unrealistic. Try again!')
        return
    current_estimate, upper, lower, conf = get_current_price_estimate(rm,ptratio,chas, large_range)
    rounded_est = np.around(current_estimate,-3)
    rounded_hi = np.around(upper, -3)
    rounded_low = np.around(lower, -3)
    print(f'The Estimated property value is ${rounded_est}')
    print(f'At {conf}% confidence level, the valuation range is')
    print(f'${rounded_low} at the lower end and ${rounded_hi} at the higher end.')

In [95]:
get_dollar_estimate(2, 30, 1.0)

The Estimated property value is $278000.0
At 95% confidence level, the valuation range is
$191000.0 at the lower end and $404000.0 at the higher end.


C:\Users\ryan6\anaconda3\Lib\site-packages\sklearn\base.py:464: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [102]:
import Boston_Valuation as val
val.get_dollar_estimate(6,12,1.0)

The Estimated property value is $783000.0
At 95% confidence level, the valuation range is
$538000.0 at the lower end and $1139000.0 at the higher end.


C:\Users\ryan6\anaconda3\Lib\site-packages\sklearn\base.py:464: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
